In [10]:
import pandas as pd
import psycopg2
import requests
from sklearn.svm import SVC
import mlflow
import os

In [5]:
conn = psycopg2.connect(
    "host=10.43.101.166 port=5433 dbname=rawdata user=admin password=admin"
)

In [ ]:
# Read table directly into a DataFrame
df = pd.read_sql_table('table_name', conn)

In [6]:
conn.close()

In [2]:
API_URL = "http://10.43.101.108/data"
params = {"group_number": 3, "day": "Tuesday"}
try:
    response = requests.get(API_URL, params=params, timeout=60)
    response.raise_for_status()
    
    # Procesar respuesta JSON
    json_data = response.json()
    tabla = json_data.get("data", [])

except requests.exceptions.HTTPError as http_err:
    # Try to get error details from JSON response

    error_data = response.json()
    print(f"API Error: {error_data.get('message', 'No error message provided')}")
    print(f"Status Code: {response.status_code}")
    if 'details' in error_data:
        print(f"Details: {error_data['details']}")


In [3]:
df = pd.DataFrame(tabla).drop(['status','city','state','prev_sold_date'], axis=1)
df

,brokered_by,price,bed,bath,acre_lot,street,zip_code,house_size
0,92147.0,110000.0,7.0,3.0,0.09,1842706.0,949.0,1192.0
1,91020.0,215000.0,2.0,1.0,0.91,1062364.0,6016.0,960.0
2,10585.0,144900.0,2.0,1.0,0.36,765673.0,6066.0,860.0
3,22611.0,239900.0,3.0,1.0,1.43,1244868.0,6016.0,1351.0
4,75650.0,249900.0,3.0,1.0,0.23,114997.0,6082.0,1220.0
...,...,...,...,...,...,...,...,...
94546,22590.0,185000.0,3.0,1.0,0.31,1247691.0,99801.0,1000.0
94547,56985.0,99900.0,1.0,1.0,0.11,669105.0,99801.0,1114.0
94548,21688.0,200000.0,3.0,2.0,0.45,410859.0,99901.0,1461.0
94549,4485.0,285000.0,2.0,2.0,0.19,371997.0,99901.0,1064.0


In [14]:
IP_MLFLOW = "http://10.43.101.168:30500"

os.environ['MLFLOW_S3_ENDPOINT_URL'] = "http://10.43.101.168:30900"
os.environ['AWS_ACCESS_KEY_ID'] = "minioadmin"
os.environ['AWS_SECRET_ACCESS_KEY'] = "minioadmin123"

mlflow.set_tracking_uri(IP_MLFLOW)

In [15]:
client = mlflow.tracking.MlflowClient()

experiment_name = "argocd_experiment"
experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    mlflow.create_experiment(experiment_name)
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='s3://mlflow/1', creation_time=1748401450491, experiment_id='1', last_update_time=1748401450491, lifecycle_stage='active', name='argocd_experiment', tags={}>

In [16]:
with mlflow.start_run(run_name="svm_training") as run:
# inicializar svm
        svm = SVC()
        model_name = "svm-model"
        
        X = df.drop(['price'],axis=1).iloc[0:1000]
        y = df['price'].iloc[0:1000]
        
                # buscar hiperparametros mas optimos
        print('Iniciando entrenamiento')
        svm = SVC(C=1.0, kernel='rbf', gamma='scale', probability=True)  # Ajusta los valores si lo deseas
        svm.fit(X, y)
        print('Entrenamiento finalizado')

        # mlflow.set_tag("column_names", ",".join(columns))
        mlflow.sklearn.log_model(
            sk_model=svm,
            artifact_path="svm",
            registered_model_name=model_name
        )

Iniciando entrenamiento
Entrenamiento finalizado


2025/05/28 03:10:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/05/28 03:10:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'svm-model'.
2025/05/28 03:10:32 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: svm-model, version 1
Created version '1' of model 'svm-model'.
2025/05/28 03:10:32 INFO mlflow.tracking._tracking_service.client: 🏃 View run svm_training at: http://10.43.101.168:30500/#/experiments/1/runs/fa162615bc1b4e15a52b40d2f1f54df3.
2025/05/28 03:10:32 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://10.43.101.168:30500/#/experiments/1.
